In [1]:
print("IBM Datathon backend setup working!")

IBM Datathon backend setup working!


In [2]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files (x86)\Microsoft Visual Studio\Shared\Python39_64\python.exe -m pip install --upgrade pip' command.


In [3]:
import fastapi
import uvicorn
from PIL import Image
import yaml

print("Backend dependencies imported successfully!")

Backend dependencies imported successfully!


In [4]:
from pathlib import Path

for folder in ["data", "models", "outputs"]:
    path = Path(folder)
    print(f"{folder}: {'Exists' if path.is_dir() else 'Missing'}")

data: Exists
models: Exists
outputs: Exists


In [5]:
from pathlib import Path

print("Project location:", Path.cwd())

Project location: c:\STUDY\IBM_Datathon_Backend


In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Backend paths configured!")
print("Data:", DATA_DIR)
print("Models:", MODEL_DIR)
print("Outputs:", OUTPUT_DIR)

Backend paths configured!
Data: c:\STUDY\IBM_Datathon_Backend\data
Models: c:\STUDY\IBM_Datathon_Backend\models
Outputs: c:\STUDY\IBM_Datathon_Backend\outputs


In [7]:
def check_dataset_folders():
    folders = {
        "Synthetic images": DATA_DIR / "synthetic" / "images",
        "ID masks": DATA_DIR / "synthetic" / "id_masks",
        "YOLO labels": DATA_DIR / "synthetic" / "yolo_labels",
        "Mapping": DATA_DIR / "synthetic" / "mapping",
        "Real test images": DATA_DIR / "real_test" / "images",
    }

    for name, folder in folders.items():
        if not folder.exists():
            print(f"{name}: MISSING FOLDER")
        elif not any(folder.iterdir()):
            print(f"{name}: WAITING FOR FILES")
        else:
            print(f"{name}: FILES RECEIVED")

In [8]:
check_dataset_folders()

Synthetic images: WAITING FOR FILES
ID masks: WAITING FOR FILES
YOLO labels: WAITING FOR FILES
Mapping: WAITING FOR FILES
Real test images: MISSING FOLDER


In [9]:
from pathlib import Path
from PIL import Image

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def validate_images(folder):
    folder = Path(folder)

    if not folder.is_dir():
        print("ERROR: Image folder does not exist.")
        return

    images = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if not images:
        print("WAITING: No image files received yet.")
        return

    valid = 0
    errors = []

    for path in images:
        try:
            with Image.open(path) as img:
                img.verify()
            valid += 1
        except Exception:
            errors.append(path.name)

    print("Images found:", len(images))
    print("Readable images:", valid)
    print("Unreadable images:", len(errors))

    for name in errors:
        print("ERROR:", name)

    print("Status:", "PASSED" if not errors else "FAILED")

In [10]:
import math

def validate_yolo_dataset(images_dir, labels_dir):
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    if not images_dir.is_dir() or not labels_dir.is_dir():
        print("ERROR: Image or label folder is missing.")
        return

    images = [
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(labels_dir.glob("*.txt"))

    if not images and not labels:
        print("WAITING: No images or labels received yet.")
        return

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    errors = []

    for stem in sorted(image_stems - label_stems):
        errors.append(f"Missing label for {stem}")

    for stem in sorted(label_stems - image_stems):
        errors.append(f"Label without matching image: {stem}.txt")

    for label in labels:
        try:
            lines = label.read_text(encoding="utf-8").splitlines()

            for number, line in enumerate(lines, 1):
                if not line.strip():
                    continue

                values = line.split()

                if len(values) != 5:
                    errors.append(
                        f"{label.name}:{number}: Expected 5 values"
                    )
                    continue

                try:
                    class_value = float(values[0])
                    box = [float(v) for v in values[1:]]

                    if not math.isfinite(class_value):
                        raise ValueError()

                    if not class_value.is_integer() or class_value < 0:
                        errors.append(
                            f"{label.name}:{number}: Invalid class ID"
                        )

                    if not all(math.isfinite(v) for v in box):
                        raise ValueError()

                    cx, cy, width, height = box

                    if not all(0 <= v <= 1 for v in box):
                        errors.append(
                            f"{label.name}:{number}: "
                            "Values outside [0, 1]"
                        )

                    if width <= 0 or height <= 0:
                        errors.append(
                            f"{label.name}:{number}: "
                            "Invalid box dimensions"
                        )

                    if (
                        cx - width / 2 < -1e-6
                        or cx + width / 2 > 1 + 1e-6
                        or cy - height / 2 < -1e-6
                        or cy + height / 2 > 1 + 1e-6
                    ):
                        errors.append(
                            f"{label.name}:{number}: "
                            "Box exceeds image boundaries"
                        )

                except ValueError:
                    errors.append(
                        f"{label.name}:{number}: Invalid numeric values"
                    )

        except OSError:
            errors.append(f"Cannot read {label.name}")

    print("Images found:", len(images))
    print("Label files found:", len(labels))
    print("Errors found:", len(errors))

    for error in errors:
        print("ERROR:", error)

    print("Status:", "PASSED" if not errors else "FAILED")

In [11]:
validate_images(DATA_DIR / "synthetic" / "images")

WAITING: No image files received yet.


In [12]:
validate_yolo_dataset(
    DATA_DIR / "synthetic" / "images",
    DATA_DIR / "synthetic" / "yolo_labels"
)


WAITING: No images or labels received yet.


In [13]:
TEST_ROOT = OUTPUT_DIR / "validation_test"
TEST_IMAGES = TEST_ROOT / "images"
TEST_LABELS = TEST_ROOT / "labels"

TEST_IMAGES.mkdir(parents=True, exist_ok=True)
TEST_LABELS.mkdir(parents=True, exist_ok=True)

print("Temporary testing folders created!")

Temporary testing folders created!


In [14]:
from pathlib import Path
from PIL import Image

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}


def validate_images(folder):
    folder = Path(folder)

    if not folder.is_dir():
        print(f"ERROR: Image folder does not exist: {folder}")
        return {
            "status": "ERROR",
            "total": 0,
            "valid": 0,
            "invalid": []
        }

    images = sorted(
        path for path in folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    if not images:
        print("WAITING: No image files received yet.")
        return {
            "status": "WAITING",
            "total": 0,
            "valid": 0,
            "invalid": []
        }

    valid_images = []
    invalid_images = []

    for path in images:
        try:
            # Check the image file's structure.
            with Image.open(path) as image:
                image.verify()

            # Reopen and fully decode the image.
            with Image.open(path) as image:
                image.load()

                if image.width <= 0 or image.height <= 0:
                    raise ValueError("Invalid image dimensions")

            valid_images.append(path.name)

        except Exception as error:
            invalid_images.append({
                "file": path.name,
                "reason": str(error) or type(error).__name__
            })

    status = "PASSED" if not invalid_images else "FAILED"

    print("\nIMAGE VALIDATION REPORT")
    print("-----------------------")
    print("Folder:", folder)
    print("Total images:", len(images))
    print("Readable images:", len(valid_images))
    print("Unreadable images:", len(invalid_images))

    for item in invalid_images:
        print(f"ERROR: {item['file']} — {item['reason']}")

    print("Status:", status)

    return {
        "status": status,
        "total": len(images),
        "valid": len(valid_images),
        "invalid": invalid_images
    }

In [15]:
image_report = validate_images(
    DATA_DIR / "synthetic" / "images"
)

WAITING: No image files received yet.


In [16]:
import math
from pathlib import Path


def read_yolo_label(label_path):
    label_path = Path(label_path)
    rows = []
    errors = []

    try:
        lines = label_path.read_text(
            encoding="utf-8"
        ).splitlines()
    except (OSError, UnicodeError) as error:
        return [], [f"Cannot read {label_path.name}: {error}"]

    for line_number, line in enumerate(lines, start=1):
        if not line.strip():
            continue

        values = line.split()

        if len(values) != 5:
            errors.append(
                f"{label_path.name}:{line_number}: "
                "Expected 5 values"
            )
            continue

        try:
            numbers = [float(value) for value in values]
        except ValueError:
            errors.append(
                f"{label_path.name}:{line_number}: "
                "Non-numeric value"
            )
            continue

        if not all(math.isfinite(value) for value in numbers):
            errors.append(
                f"{label_path.name}:{line_number}: "
                "Non-finite numeric value"
            )
            continue

        class_id, cx, cy, width, height = numbers

        rows.append(
            (line_number, class_id, cx, cy, width, height)
        )

    return rows, errors

In [17]:
def check_yolo_values(label_name, rows, class_names=None):
    errors = []

    for line_number, class_id, cx, cy, width, height in rows:
        if not class_id.is_integer() or class_id < 0:
            errors.append(
                f"{label_name}:{line_number}: Invalid class ID"
            )
            continue

        class_id = int(class_id)

        if class_names is not None and class_id >= len(class_names):
            errors.append(
                f"{label_name}:{line_number}: "
                f"Class ID {class_id} is not defined"
            )

        if not all(
            0 <= value <= 1
            for value in (cx, cy, width, height)
        ):
            errors.append(
                f"{label_name}:{line_number}: "
                "Coordinates must be between 0 and 1"
            )
            continue

        if width <= 0 or height <= 0:
            errors.append(
                f"{label_name}:{line_number}: "
                "Bounding-box width and height must be positive"
            )
            continue

        if (
            cx - width / 2 < -1e-6
            or cx + width / 2 > 1 + 1e-6
            or cy - height / 2 < -1e-6
            or cy + height / 2 > 1 + 1e-6
        ):
            errors.append(
                f"{label_name}:{line_number}: "
                "Bounding box extends beyond image boundaries"
            )

    return errors

In [18]:
IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp", ".webp"
}


def validate_yolo_dataset(images_dir, labels_dir, class_names=None):
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    if not images_dir.is_dir():
        return {
            "status": "ERROR",
            "images": 0,
            "labels": 0,
            "errors": [f"Missing image folder: {images_dir}"]
        }

    if not labels_dir.is_dir():
        return {
            "status": "ERROR",
            "images": 0,
            "labels": 0,
            "errors": [f"Missing label folder: {labels_dir}"]
        }

    images = sorted(
        path for path in images_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    labels = sorted(labels_dir.glob("*.txt"))

    if not images and not labels:
        return {
            "status": "WAITING",
            "images": 0,
            "labels": 0,
            "errors": []
        }

    image_stems = {path.stem for path in images}
    label_stems = {path.stem for path in labels}
    errors = []

    for stem in sorted(image_stems - label_stems):
        errors.append(f"Missing label for image: {stem}")

    for stem in sorted(label_stems - image_stems):
        errors.append(f"Label has no matching image: {stem}.txt")

    for label_path in labels:
        rows, format_errors = read_yolo_label(label_path)
        errors.extend(format_errors)

        value_errors = check_yolo_values(
            label_path.name,
            rows,
            class_names=class_names
        )
        errors.extend(value_errors)

    status = "FAILED" if errors else "PASSED"

    return {
        "status": status,
        "images": len(images),
        "labels": len(labels),
        "error_count": len(errors),
        "errors": errors
    }

In [19]:
def validate_full_dataset(images_dir, labels_dir, class_names=None):
    print("========== IMAGE VALIDATION ==========")
    image_report = validate_images(images_dir)

    print("\n========== YOLO DATASET VALIDATION ==========")
    label_report = validate_yolo_dataset(
        images_dir,
        labels_dir,
        class_names=class_names
    )

    statuses = [
        image_report["status"],
        label_report["status"]
    ]

    if "ERROR" in statuses:
        overall_status = "ERROR"
    elif "FAILED" in statuses:
        overall_status = "FAILED"
    elif all(status == "PASSED" for status in statuses):
        overall_status = "PASSED"
    else:
        overall_status = "WAITING"

    print("\n========== FINAL DATASET REPORT ==========")
    print("Image status:", image_report["status"])
    print("Label status:", label_report["status"])
    print("Overall status:", overall_status)

    if label_report.get("errors"):
        print("\nDataset errors:")
        for error in label_report["errors"]:
            print("-", error)

    return {
        "status": overall_status,
        "image_report": image_report,
        "label_report": label_report
    }

In [20]:
CLASS_NAMES = None

classes_file = MODEL_DIR / "classes.txt"

if classes_file.is_file():
    CLASS_NAMES = [
        line.strip()
        for line in classes_file.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

final_report = validate_full_dataset(
    DATA_DIR / "synthetic" / "images",
    DATA_DIR / "synthetic" / "yolo_labels",
    class_names=CLASS_NAMES
)

========== IMAGE VALIDATION ==========
WAITING: No image files received yet.

========== YOLO DATASET VALIDATION ==========

========== FINAL DATASET REPORT ==========
Image status: WAITING
Label status: WAITING
Overall status: WAITING


In [21]:
def validate_image_mask_pairs(images_dir, masks_dir):
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)

    if not images_dir.is_dir() or not masks_dir.is_dir():
        print("ERROR: Image folder or ID-mask folder is missing.")
        return {"status": "ERROR", "errors": []}

    images = sorted(
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    masks = sorted(
        p for p in masks_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    if not images and not masks:
        print("WAITING: No image-mask files received yet.")
        return {"status": "WAITING", "errors": []}

    image_map = {}
    mask_map = {}
    errors = []

    for path in images:
        if not path.stem.startswith("img_"):
            errors.append(f"Unexpected image filename: {path.name}")
        else:
            image_map[path.stem[4:]] = path

    for path in masks:
        if not path.stem.startswith("ids_"):
            errors.append(f"Unexpected ID-mask filename: {path.name}")
        else:
            mask_map[path.stem[4:]] = path

    for key in sorted(image_map.keys() - mask_map.keys()):
        errors.append(f"Missing ID mask for {image_map[key].name}")

    for key in sorted(mask_map.keys() - image_map.keys()):
        errors.append(f"ID mask has no matching image: {mask_map[key].name}")

    checked_pairs = 0

    for key in sorted(image_map.keys() & mask_map.keys()):
        image_path = image_map[key]
        mask_path = mask_map[key]

        try:
            with Image.open(image_path) as image:
                image.verify()
            with Image.open(mask_path) as mask:
                mask.verify()

            with Image.open(image_path) as image:
                image_size = image.size
            with Image.open(mask_path) as mask:
                mask_size = mask.size

            checked_pairs += 1

            if image_size != mask_size:
                errors.append(
                    f"Dimension mismatch: {image_path.name} "
                    f"{image_size} vs {mask_path.name} {mask_size}"
                )

        except Exception as error:
            errors.append(
                f"Unreadable pair {image_path.name} / "
                f"{mask_path.name}: {error}"
            )

    status = "FAILED" if errors else "PASSED"

    print("Image files:", len(images))
    print("ID masks:", len(masks))
    print("Matched pairs checked:", checked_pairs)
    print("Errors:", len(errors))

    for error in errors:
        print("ERROR:", error)

    print("Status:", status)

    return {
        "status": status,
        "images": len(images),
        "masks": len(masks),
        "checked_pairs": checked_pairs,
        "errors": errors
    }

In [22]:
mask_pair_report = validate_image_mask_pairs(
    DATA_DIR / "synthetic" / "images",
    DATA_DIR / "synthetic" / "id_masks"
)

WAITING: No image-mask files received yet.


In [23]:
def inspect_id_masks(folder):
    folder = Path(folder)

    if not folder.is_dir():
        print("ERROR: ID-mask folder is missing.")
        return

    masks = sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    if not masks:
        print("WAITING: No ID masks received yet.")
        return

    for path in masks:
        try:
            with Image.open(path) as mask:
                mask.load()
                colors = mask.getcolors(maxcolors=100_000)

                print(f"\nFile: {path.name}")
                print("Mode:", mask.mode)
                print("Dimensions:", mask.size)

                if colors is None:
                    print("Distinct pixel values: more than 100,000")
                else:
                    print("Distinct pixel values:", len(colors))

        except Exception as error:
            print(f"ERROR: {path.name}: {error}")

In [24]:
inspect_id_masks(DATA_DIR / "synthetic" / "id_masks")


WAITING: No ID masks received yet.
